# 09 — Benchmark Stato dell'Arte (SOTA) & Scalabilità su Sequenze Lunghe (eccDNA 500 bp -> SARS-CoV-2 30 kb -> 500 kb)

## Obiettivi del Notebook
Confrontare in modo rigoroso le nostre varianti FCGR ottimizzate contro la **Baseline originale** e le principali **implementazioni dello Stato dell'Arte (SOTA)** in letteratura:
1. **Baseline Originale**: doppio ciclo Python (`compute_fcgr`, `float32`).
2. **SOTA 1 — `complexCGR`** (*Avila Cartes, Bonizzoni, Della Vedova et al. — CouGaR-g su SARS-CoV-2*): libreria Python ufficiale di riferimento.
3. **SOTA 2 — `Dictionary / Counter FCGR`** (*stile DeLUCS — Millán Arias et al. / Randhawa et al.*): conteggio tramite dizionario `Counter` + mappatura CGR sui k-mer distinti.
4. **SOTA 3 — `Positional FCGR (PFCGR)`** (*Avila Cartes et al.*): rappresentazione a 3 canali `float32` (Frequenza, Posizione Media, Std Posizione).
5. **Le nostre varianti (`src/fcgr.py`)**: `Bitwise Pure (uint8)`, `Opt 1 (RC+Log, uint8)`, `Opt 2 (Multi-Scala, uint8)`, `Opt 3b (O/E Enrichment + Multi-Scala, uint8)`.

### Perché aumentiamo la lunghezza $L$ fino a `30.000 bp` (SARS-CoV-2) e `500.000 bp`?
Su frammenti corti (`~500 bp`), i tempi per singola sequenza sono di pochi millisecondi e la differenza visiva sembra piccola. Scalando $L$ a **`30 kb` (genoma SARS-CoV-2)** e **`500 kb`**, e proiettando su dataset reali (`100.000` genomi virali / `3.75M` eccDNA), emerge in modo inequivocabile il passaggio da **ore / giorni di calcolo** a **pochi minuti**, insieme al crollo della sparsità al crescere di $L$.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Installazione libreria SOTA ufficiale `complexcgr` (CouGaR-g / Avila Cartes et al.)
import importlib.util
if importlib.util.find_spec('complexcgr') is None:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'complexcgr'], check=False)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
COMPARISON_DIR = PROJECT_ROOT / 'artifacts' / 'final_fcgr_comparison'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

from src.fcgr import (
    compute_fcgr,
    normalize_fcgr,
    compute_fcgr_vectorized,
    compute_fcgr_sota_dict,
    compute_fcgr_sota_pfcgr,
    normalize_fcgr_log,
    compute_multiscale_fcgr,
    compute_opt3_enriched_fcgr,
    quantize_fcgr_uint8,
    dequantize_fcgr_uint8_prob,
)

try:
    from complexcgr import FCGR as ComplexCGR_FCGR
    HAS_COMPLEXCGR = True
    print('Libreria SOTA `complexcgr` caricata con successo!')
except ImportError:
    HAS_COMPLEXCGR = False
    print('Nota: `complexcgr` non disponibile; il benchmark proseguirà con le altre varianti SOTA.')

## 1. Benchmark di Scalabilità per Lunghezza Sequenza $L$ (`500 bp` $\rightarrow$ `5 kb` $\rightarrow$ `30 kb SARS-CoV-2` $\rightarrow$ `100 kb` $\rightarrow$ `500 kb`)
Misuriamo il tempo medio per sequenza (`ms`), lo **Speedup** rispetto alla Baseline e rispetto a **SOTA `complexCGR` / `Dict`**, e la memoria occupata (`KB`) a $k=6$ e $k=7$.

In [ ]:
rng = np.random.default_rng(42)
LENGTHS = [500, 5_000, 30_000, 100_000, 500_000]
LABELS_L = {500: '500 bp (eccDNA)', 5_000: '5 kb', 30_000: '30 kb (SARS-CoV-2)', 100_000: '100 kb', 500_000: '500 kb'}
TEST_SEQS = {L: ''.join(rng.choice(list('ACGT'), size=L)) for L in LENGTHS}

def time_fn(fn, n_reps: int) -> float:
    fn()  # warm-up
    t0 = time.perf_counter()
    for _ in range(n_reps):
        fn()
    return (time.perf_counter() - t0) * 1000.0 / n_reps

bench_rows = []
for k_val in [6, 7]:
    ccgr = ComplexCGR_FCGR(k_val) if HAS_COMPLEXCGR else None
    side = 1 << k_val
    kb_1ch_f32 = (side * side * 4) / 1024.0
    kb_1ch_u8 = (side * side * 1) / 1024.0
    kb_3ch_f32 = (3 * side * side * 4) / 1024.0
    kb_3ch_u8 = (3 * side * side * 1) / 1024.0

    for L in LENGTHS:
        seq = TEST_SEQS[L]
        n_fast = 20 if L <= 30_000 else (5 if L <= 100_000 else 2)
        n_slow = 5 if L <= 5_000 else (2 if L <= 30_000 else 1)

        # 1. Baseline Python loop (skip 500k se troppo lenta, o 1 rep)
        t_base = time_fn(lambda: normalize_fcgr(compute_fcgr(seq, k_val, False)[0]), n_slow if L <= 100_000 else 1)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '1_Baseline_Python_f32', 'time_ms': t_base, 'mem_KB': kb_1ch_f32, 'channels': 1})

        # 2. SOTA: complexCGR
        if HAS_COMPLEXCGR:
            t_ccgr = time_fn(lambda: ccgr(seq), n_slow if L <= 100_000 else 1)
            bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '2_SOTA_complexCGR_f64', 'time_ms': t_ccgr, 'mem_KB': kb_1ch_f32 * 2, 'channels': 1})

        # 3. SOTA: Dictionary / Counter (DeLUCS / Randhawa style)
        t_dict = time_fn(lambda: normalize_fcgr(compute_fcgr_sota_dict(seq, k_val, False)[0]), n_slow if L <= 100_000 else 1)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '3_SOTA_DictCounter_f32', 'time_ms': t_dict, 'mem_KB': kb_1ch_f32, 'channels': 1})

        # 4. SOTA: Positional FCGR (PFCGR, 3ch f32)
        t_pfcgr = time_fn(lambda: compute_fcgr_sota_pfcgr(seq, k_val), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '4_SOTA_PFCGR_3ch_f32', 'time_ms': t_pfcgr, 'mem_KB': kb_3ch_f32, 'channels': 3})

        # 5. Nostro: Bitwise Pure + uint8
        t_bit = time_fn(lambda: quantize_fcgr_uint8(normalize_fcgr(compute_fcgr_vectorized(seq, k_val, False)[0])), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '5_Ours_Bitwise_u8', 'time_ms': t_bit, 'mem_KB': kb_1ch_u8, 'channels': 1})

        # 6. Nostro: Opt 1 (RC + Log, uint8)
        t_opt1 = time_fn(lambda: quantize_fcgr_uint8(normalize_fcgr_log(compute_fcgr_vectorized(seq, k_val, False, True)[0])), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '6_Ours_Opt1_RCLog_u8', 'time_ms': t_opt1, 'mem_KB': kb_1ch_u8, 'channels': 1})

        # 7. Nostro: Opt 2 (Multi-Scala, 3ch uint8)
        ks_ms = (k_val - 2, k_val - 1, k_val)
        t_opt2 = time_fn(lambda: quantize_fcgr_uint8(compute_multiscale_fcgr(seq, ks=ks_ms, target_k=k_val)[0]), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '7_Ours_Opt2_MultiScale_3ch_u8', 'time_ms': t_opt2, 'mem_KB': kb_3ch_u8, 'channels': 3})

        # 8. Nostro: Opt 3b (O/E Enrichment + Multi-Scala, 3ch uint8)
        t_opt3b = time_fn(lambda: quantize_fcgr_uint8(compute_opt3_enriched_fcgr(seq, target_k=k_val)[0]), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '8_Ours_Opt3b_EnrichedMS_3ch_u8', 'time_ms': t_opt3b, 'mem_KB': kb_3ch_u8, 'channels': 3})

sota_bench_df = pd.DataFrame(bench_rows)
sota_bench_df.to_csv(RESULTS_DIR / 'sota_length_scaling_benchmark.csv', index=False)

print('=== BENCHMARK TEMPI (ms / sequenza) A k=6 AL VARIARE DI L (500 bp -> 500 kb) ===')
piv_k6 = sota_bench_df[sota_bench_df['k'] == 6].pivot(index='method', columns='scale', values='time_ms')[list(LABELS_L.values())]
display(piv_k6.round(3))

print('\n=== BENCHMARK TEMPI (ms / sequenza) A k=7 AL VARIARE DI L (500 bp -> 500 kb) ===')
piv_k7 = sota_bench_df[sota_bench_df['k'] == 7].pivot(index='method', columns='scale', values='time_ms')[list(LABELS_L.values())]
display(piv_k7.round(3))

## 2. Impatto Reale su Scala SARS-CoV-2 (`100.000` Genomi da `30 kb`) ed eccDNA Completo (`3.75M` Sequenze)
Qui traduciamo i millisecondi in **ore / minuti reali** e **GB di RAM** per vedere chiaramente il vantaggio pratico rispetto a Baseline e SOTA.

In [ ]:
k6_30k = sota_bench_df[(sota_bench_df['k'] == 6) & (sota_bench_df['L_bp'] == 30_000)].copy()
k7_30k = sota_bench_df[(sota_bench_df['k'] == 7) & (sota_bench_df['L_bp'] == 30_000)].copy()

N_COVID = 100_000
for label_k, sub_df in [('k=6 (64x64)', k6_30k), ('k=7 (128x128)', k7_30k)]:
    base_t = float(sub_df.loc[sub_df['method'] == '1_Baseline_Python_f32', 'time_ms'].iloc[0])
    sub_df = sub_df.copy()
    sub_df['speedup_vs_baseline_x'] = (base_t / sub_df['time_ms']).round(1)
    sub_df['100k_covid_minutes'] = ((sub_df['time_ms'] * N_COVID) / 60_000.0).round(2)
    sub_df['100k_covid_hours'] = (sub_df['100k_covid_minutes'] / 60.0).round(2)
    sub_df['100k_covid_RAM_GB'] = ((sub_df['mem_KB'] * N_COVID) / (1024 ** 2)).round(2)
    print(f'=== PROIEZIONE REALE: 100.000 GENOMI SARS-CoV-2 (30 kb) — {label_k} ===')
    display(sub_df[['method', 'time_ms', 'speedup_vs_baseline_x', '100k_covid_minutes', '100k_covid_hours', '100k_covid_RAM_GB']])

# Grafico log-log Scalabilità Tempo vs Lunghezza L (k=6)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sub6 = sota_bench_df[sota_bench_df['k'] == 6]
for m_name, grp in sub6.groupby('method'):
    lw = 2.6 if 'Ours' in m_name else 1.8
    ls = '-' if 'Ours' in m_name else '--'
    axes[0].loglog(grp['L_bp'], grp['time_ms'], marker='o', linewidth=lw, linestyle=ls, label=m_name)
axes[0].axvline(30_000, color='gray', linestyle=':', alpha=0.8)
axes[0].text(32_000, axes[0].get_ylim()[0] * 2, 'SARS-CoV-2\n(30 kb)', fontsize=9, color='gray')
axes[0].set_xlabel('Lunghezza Sequenza L (bp) — Scala Log')
axes[0].set_ylabel('Tempo per Sequenza (ms) — Scala Log')
axes[0].set_title('Scalabilità Temporale (k=6): Baseline vs SOTA vs Nostre Varianti')
axes[0].legend(fontsize=8.5)
axes[0].grid(True, which='both', alpha=0.25)

# Bar chart tempo reale su 100k genomi SARS-CoV-2 (k=7, ore)
k7_plot = k7_30k.copy()
k7_plot['minutes'] = (k7_plot['time_ms'] * N_COVID) / 60_000.0
colors = ['#C53030' if 'Baseline' in m else ('#DD6B20' if 'SOTA' in m else '#2F855A') for m in k7_plot['method']]
axes[1].barh(k7_plot['method'], k7_plot['minutes'], color=colors)
axes[1].set_xscale('log')
axes[1].set_xlabel('Minuti per 100.000 Genomi SARS-CoV-2 (30 kb, k=7) — Scala Log')
axes[1].set_title('Tempo Stimato su 100k Genomi COVID (k=7)')
for idx_r, (_, row) in enumerate(k7_plot.iterrows()):
    axes[1].text(row['minutes'] * 1.15, idx_r, f"{row['minutes']:.1f} min", va='center', fontsize=8.5, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'sota_scaling_benchmark_covid.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Analisi della Sparsità al variare di $L$ (`500 bp` $\rightarrow$ `30 kb` $\rightarrow$ `500 kb`) e $k \in [4, 10]$
Questa mappa spiega il ponte biologico-matematico tra **eccDNA (`~500 bp`)** e **SARS-CoV-2 (`~30.000 bp`)**:
- Su eccDNA (`500 bp`), già a $k=6$ la matrice è vuota all'`~88%` e a $k=7$ al `97%`.
- Su SARS-CoV-2 (`30.000 bp`), a $k=6$ la matrice è **quasi completamente piena (`~0.1%` sparsità!)**, a $k=7$ è densa (`~16%` sparsità) e perfino a $k=8$ resta informativa (`~63%` sparsità)!

In [ ]:
sp_rows = []
for k_val in range(4, 11):
    total_cells = (1 << k_val) ** 2
    for L in LENGTHS:
        _, _, distinct = compute_fcgr_vectorized(TEST_SEQS[L], k_val, return_distinct=True)
        sparsity_pct = (1.0 - distinct / float(total_cells)) * 100.0
        sp_rows.append({'k': k_val, 'scale': LABELS_L[L], 'L_bp': L, 'sparsity_pct': round(sparsity_pct, 2)})

sp_df = pd.DataFrame(sp_rows)
sp_piv = sp_df.pivot(index='k', columns='scale', values='sparsity_pct')[list(LABELS_L.values())]
sp_piv.to_csv(RESULTS_DIR / 'sparsity_by_k_and_length_sota.csv')
print('=== SPARSITA\' (%) AL VARIARE DI k (4..10) E LUNGHEZZA L (500 bp -> 500 kb) ===')
display(sp_piv)

fig, ax = plt.subplots(figsize=(10, 5))
for L in LENGTHS:
    sub = sp_df[sp_df['L_bp'] == L]
    lw = 3.0 if L in (500, 30_000) else 1.8
    ax.plot(sub['k'], sub['sparsity_pct'], marker='o', linewidth=lw, label=LABELS_L[L])
ax.axhline(90.0, linestyle='--', color='gray', alpha=0.6, label='Soglia Critica 90% Sparsità')
ax.set_xlabel('Ordine k-mer (k)')
ax.set_ylabel('Sparsità (% Celle Vuote)')
ax.set_title('Transizione di Regime della Sparsità: eccDNA (500 bp) vs SARS-CoV-2 (30 kb)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'sparsity_regime_eccdna_vs_covid.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Confronto Visivo su Scala Virale (`30.000 bp`, $k=7$): SOTA `Positional FCGR (PFCGR)` vs Nostra `Opt 3b`

In [ ]:
seq_30k = TEST_SEQS[30_000]
pfcgr_30k, _ = compute_fcgr_sota_pfcgr(seq_30k, k=7)
opt3b_30k, _ = compute_opt3_enriched_fcgr(seq_30k, target_k=7)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
pfcgr_titles = ['SOTA PFCGR Ch0: Frequenza (k=7)', 'SOTA PFCGR Ch1: Posizione Media', 'SOTA PFCGR Ch2: Std Posizione']
opt3b_titles = ['Ours Opt3b Ch0: Log-Prob (k=7)', 'Ours Opt3b Ch1: O/E Enrichment (k=7)', 'Ours Opt3b Ch2: Multi-Scala k=5+6']

for i in range(3):
    im_top = axes[0, i].imshow(np.log1p(pfcgr_30k[i] * 1000.0) if i == 0 else pfcgr_30k[i], cmap='viridis', origin='lower', interpolation='nearest')
    axes[0, i].set_title(pfcgr_titles[i], fontsize=10)
    plt.colorbar(im_top, ax=axes[0, i], fraction=0.046, pad=0.04)

    im_bot = axes[1, i].imshow(np.log1p(opt3b_30k[i] * 1000.0), cmap='magma' if i == 1 else 'plasma', origin='lower', interpolation='nearest')
    axes[1, i].set_title(opt3b_titles[i], fontsize=10)
    plt.colorbar(im_bot, ax=axes[1, i], fraction=0.046, pad=0.04)

plt.suptitle('Confronto Visivo su Sequenza Lunghezza SARS-CoV-2 (30.000 bp, k=7, 128x128): SOTA PFCGR (192 KB) vs Ours Opt3b (48 KB uint8)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'visual_sota_pfcgr_vs_opt3b_30kb.png', dpi=300, bbox_inches='tight')
plt.show()